<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Solution: Detect what the query asks for

**[Detect what the query asks for](https://learning.nextia-ai.com/courses/search/m04/detect-what-the-query-asks-for/)** · Search and Ranking: Find the Best Match · Module 4, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** detect a price limit and colours with rules, write a small Naive Bayes classifier that guesses the product class of a query, and measure how often its guess is right.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 10 MB). No account, no key and no model download. |
| **You should know** | Synonym rules, from [Synonyms and query rewriting](https://learning.nextia-ai.com/courses/search/m04/synonyms-and-query-rewriting/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m04/detect-what-the-query-asks-for/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M04-L04-detect-what-the-query-asks-for/detect-what-the-query-asks-for-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 10 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Rules for a price limit and colours

A **regular expression** is a pattern that describes text. The project's pattern finds a limit word ("under", "moins de" …), maybe a "$", a number, then maybe "$", "dollars" or "cad". Colours come from a list of words. Run the next cell.

In [ ]:
import re, unicodedata
from collections import Counter

def fold(text):
    t = unicodedata.normalize("NFKD", str(text).lower())
    return "".join(c for c in t if not unicodedata.combining(c))

def plain(text):
    return [t for t in re.split(r"[^\w]+|_", fold(text)) if t]

# a limit word, maybe "$", a number, then maybe "$", "dollars" or "cad" (the project's pattern)
PRICE = re.compile(r"(?:under|below|less than|max|cheaper than|moins de|sous)\s*\$?\s*(\d+(?:\.\d+)?)\s*"
                   r"(?:\$|dollars?|cad)?")
COLOURS = {"black", "white", "grey", "gray", "blue", "navy", "green", "red", "pink", "yellow", "orange", "purple",
           "brown", "beige", "gold", "silver", "bronze", "teal", "ivory", "cream", "tan", "natural", "turquoise",
           "noir", "blanc", "gris", "bleu", "vert", "rouge", "rose", "jaune", "brun", "dore", "argent"}

def detect_rules(text):
    t = fold(text)
    m = PRICE.search(t)
    clean = PRICE.sub(" ", t).strip() if m else t
    return {"price_max": float(m.group(1)) if m else None, "colours": [w for w in plain(clean) if w in COLOURS],
            "clean": clean}

for text in ["grey cushion under 50 dollars", "coussin gris moins de 50 $", "red rug for balcony", "sofa"]:
    print(f"{text:32} {detect_rules(text)}")
q = queries()
with_price = int(q["query"].map(lambda t: PRICE.search(fold(t)) is not None).sum())
print(with_price, "of", len(q), "queries in the query set ask for a price")

> **Check.** "under 50 dollars" and "moins de 50 $" give a price limit of 50, and the price words are removed from the query. But 0 of the 624 queries ask for a price: the test set cannot measure a price filter. A test set measures only the queries that it contains.

## 2. A classifier for the product class

No list of words covers hundreds of classes, so the classifier learns from the catalogue. Every product is an example of its class. The next cell counts how often each word appears in the titles (English and French) and the class name of each class. It takes some seconds.

In [ ]:
cat = catalogue()
print((cat["product_class"] == "").sum(), "products have no class: they are left out")
cat = cat[cat["product_class"] != ""]
docs = [plain(f"{t} {f} {c}") for t, f, c in zip(cat["title"], cat["title_fr"], cat["product_class"])]
df = Counter(w for d in docs for w in set(d))
vocab = {w: i for i, w in enumerate(sorted(w for w, n in df.items() if n >= 2))}   # words in 2+ products
classes = sorted(cat["product_class"].unique())
cid = {c: i for i, c in enumerate(classes)}
counts = np.zeros((len(classes), len(vocab)), dtype=np.float32)   # how often each word is in each class
for d, c in zip(docs, cat["product_class"]):
    ws = [vocab[w] for w in d if w in vocab]
    np.add.at(counts[cid[c]], ws, 1)
# the share of each word among the words of its class, with 1 added to every count (so no share is 0), as a logarithm
log_share = np.log((counts + 1) / (counts.sum(axis=1, keepdims=True) + len(vocab)))
log_prior = np.log(cat["product_class"].value_counts().reindex(classes).to_numpy() / len(cat))
print(len(classes), "classes,", len(vocab), "words")

> **Check.** 547 classes and 10172 words. `log_share[c, w]` is the logarithm of the share of word w among the words of class c. Adding logarithms is the same as multiplying the shares, and it avoids numbers too small for the computer.

## 3. Your turn: guess the class

**Naive Bayes** guesses the class whose words best explain the query: for each class, the log prior (how common the class is) plus the log shares of the query's words. It is "naive" because it treats every word as independent of the others. Complete `total`.

In [ ]:
def guess(text):
    """The class with the highest log prior + sum of the log shares of the query's known words."""
    ws = [vocab[w] for w in plain(text) if w in vocab]
    total = log_prior + log_share[:, ws].sum(axis=1)
    return classes[int(np.argmax(total))]

for text in ["red rug", "palram harmony greenhouses", "clear shower curtain", "farmhouse cabinet", "canape"]:
    print(f"{text:28} -> {guess(text)}")

In [ ]:
expect('guess("red rug")', guess("red rug"), 'Area Rugs')

> **Check.** "red rug" is Area Rugs. "clear shower curtain" gives "Shower Curtain Hooks and Accessories": the words "shower" and "curtain" are in the titles of curtains and of the hooks for them, and a word-by-word classifier cannot tell which one the person wants. That is why the project uses the class only as a **boost** (it moves products up), never as a **filter** (it would remove the curtains).

This notebook writes a plain Naive Bayes on single words, so you can see every step. The project uses a variant from scikit-learn (Complement Naive Bayes, with word pairs), so some of its guesses differ.

## 4. Your turn: how often is the first guess right?

For each validation query with exact products, call the most common class of its exact products the right class. Compute the share of queries where the first guess is that class.

In [ ]:
j = judgments()
val = q[(q["split"] == "validation") & q["source"].isin(["wands", "translated", "typo"])]
klass = cat.set_index("product_id")["product_class"]   # products with a class only
rows = []
for qid, text, src in zip(val["query_id"], val["query"], val["source"]):
    exact = [p for p, g in j.get(qid, {}).items() if g == 2 and p in klass.index]
    if exact:
        truth = klass.loc[exact].value_counts().index[0]   # the most common class of the exact products
        rows.append((qid, text, src, truth, guess(text)))
res = pd.DataFrame(rows, columns=["query_id", "query", "source", "truth", "guess"])
accuracy = (res["truth"] == res["guess"]).mean()
print(len(res), "queries; first guess right:", round(accuracy, 3))

In [ ]:
expect('accuracy', accuracy, 0.468, tolerance=0.002)

> **Check.** About 0.468 on 94 queries.

> **Predict.** Which kind of query gets the right class most often: English, French or misspelt?

In [ ]:
if accuracy is ... or accuracy is None:
    print("Not yet: complete the task above first.")
else:
    res["right"] = res["truth"] == res["guess"]
    print(res.groupby("source")["right"].mean().round(3))
    print(res[~res["right"]].head(8)[["query", "truth", "guess"]].to_string(index=False))

> **Check.** English 0.520, French 0.423, misspelt 0.389. A wrong guess is common, so a boost must be measured per query, like every rule of this module.

In the project: `search compare bm25-norm+spell+syn bm25-norm+spell+syn+detect --by source`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Search by meaning](https://learning.nextia-ai.com/courses/search/m05/search-by-meaning/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m04/detect-what-the-query-asks-for/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.